# modification
1) unfreeze last 2 blocks other than FCL
2) fedprox instead of fed avg
3) no scheduler


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import copy
import numpy as np
from collections import OrderedDict

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import ImageFolder
from torchvision.transforms import Compose, Resize, ToTensor, Normalize
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights

from sklearn.metrics import classification_report


In [3]:


class Net(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()
        # Load the base EfficientNet model with pre-trained weights
        self.base = efficientnet_b3(weights=EfficientNet_B3_Weights.IMAGENET1K_V1)

        # 1. Freeze the entire backbone initially
        for p in self.base.parameters():
            p.requires_grad = False

        # 2. PARTIAL UNFREEZE: Unfreeze the last two blocks (features.7 and features.8)
        # EfficientNet features are in blocks (0 to 8). We unfreeze the last few.
        # Unfreeze features.7 (Block 7)
        for p in self.base.features[7].parameters():
            p.requires_grad = True

        # Unfreeze features.8 (Block 8, which contains the final convolution layer)
        for p in self.base.features[8].parameters():
            p.requires_grad = True

        # 3. Replace and Unfreeze the final fully connected (FC) classifier
        num_ftrs = self.base.classifier[-1].in_features
        self.base.classifier[-1] = nn.Linear(num_ftrs, num_classes)

        for p in self.base.classifier.parameters():
            p.requires_grad = True

    def forward(self, x):
        return self.base(x)


In [4]:
img_tf = Compose([
    Resize((256, 256)),
    ToTensor(),
    Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])


In [5]:
DATA_ROOT = "/content/drive/MyDrive/Datasets/Splitted_data"

def load_client_data(client_id):
    path = os.path.join(DATA_ROOT, f"Client_{client_id}")
    dataset = ImageFolder(path, transform=img_tf)

    n = len(dataset)
    t = int(0.8 * n)
    v = n - t

    train_ds, val_ds = random_split(dataset, [t, v], generator=torch.Generator().manual_seed(42))
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=32)
    return train_loader, val_loader


In [6]:
def local_train(model, loader, epochs, lr, device):
    model.to(device)
    model.train()

    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()

    for _ in range(epochs):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = crit(model(x), y)
            loss.backward()
            opt.step()

    return model.state_dict()  # return updated weights


In [7]:
from sklearn.metrics import classification_report

def local_eval(model, data_loader, device):
    model.eval()
    model.to(device)
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(data_loader)

    # Generate a detailed classification report
    report = classification_report(all_labels, all_preds, output_dict=True, zero_division=0)

    return avg_loss, report

In [8]:
def local_train_fedprox(model, loader, epochs, lr, mu, device):
    model.to(device)
    model.train()

    # Get the state_dict of the model at the start of the round (w^t)
    # This is the reference point for the proximal term.
    # We convert it to an OrderedDict to avoid referencing the model's parameters directly.
    w_global = OrderedDict(copy.deepcopy(model.state_dict()))

    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()

    for _ in range(epochs):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()

            # --- 1. Standard Loss (Cross-Entropy) ---
            loss = crit(model(x), y)

            # --- 2. Proximal Term (FedProx) ---
            proximal_term = 0.0

            # Iterate through all local model parameters (w)
            for name, param in model.named_parameters():
                if name in w_global:
                    # Calculate ||w - w^t||^2 for each layer
                    # Note: w_global[name] is already on CPU due to deepcopy,
                    # so we move it to the current device for calculation.
                    proximal_term += torch.sum((param - w_global[name].to(device)) ** 2)

            # Add the proximal term to the loss: Loss + (mu/2) * ||w - w^t||^2
            loss += (mu / 2.0) * proximal_term

            # --- 3. Backpropagation ---
            loss.backward()
            opt.step()

    return model.state_dict()  # return updated weights

In [9]:
def fed_avg(models):
    avg = copy.deepcopy(models[0])
    for k in avg.keys():
        for i in range(1, len(models)):
            avg[k] += models[i][k]
        avg[k] = avg[k] / len(models)
    return avg


In [10]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_CLIENTS = 3
ROUNDS = 10
LOCAL_EPOCHS = 10
LR = 0.0001
MU = 0.01

global_model = Net()

# Load all client validation data once
client_val_loaders = {}
for cid in range(1, NUM_CLIENTS + 1):
    _, val_loader = load_client_data(cid)
    client_val_loaders[cid] = val_loader

# Variables to track the best overall metrics
best_global_accuracy = -1.0
best_metrics_report = None
best_round_info = {"round": -1, "client_id": -1}

def print_aligned_report(report, title):
    """Prints the classification report dictionary in a nicely aligned table."""
    print(f"\n--- {title} ---")

    # Define header with appropriate fixed widths
    # class: 10, precision: 12, recall: 10, f1-score: 12, support: 10
    header = f"{'class':<10}{'precision':<12}{'recall':<10}{'f1-score':<12}{'support':<10}"
    print(header)
    print("-" * len(header))

    for label in sorted([k for k in report.keys() if k.isdigit()]):
        metrics = report[label]
        print(f"{label:<10}   {metrics['precision']:.4f}   {metrics['recall']:.4f}    {metrics['f1-score']:.4f}     {metrics['support']:<10}")
    print("-" * len(header))
    macro_avg = report['macro avg']
    weighted_avg = report['weighted avg']
    total_support = int(weighted_avg['support'])
    print(f"{'accuracy':<10}{'':<12}{'':<10}{report['accuracy']:.4f}{total_support:<10}")


print("Starting FL Training with Round-wise Validation and Best Metrics Tracking...")

for r in range(ROUNDS):
    print(f"\n----- Round {r+1} (Training) -----")

    client_weights = []

    # ----- CLIENT UPDATE (Train locally) -----
    for cid in range(1, NUM_CLIENTS + 1):
        train_loader, _ = load_client_data(cid)

        local_model = copy.deepcopy(global_model)
        updated_weights = local_train_fedprox(local_model, train_loader, LOCAL_EPOCHS, LR, MU, DEVICE)
        # updated_weights = local_train(local_model, train_loader, LOCAL_EPOCHS, LR, DEVICE)
        client_weights.append(updated_weights)
        print(f"  Client {cid} finished local training.")

    # ----- SERVER AGGREGATION -----
    new_global_weights = fed_avg(client_weights)
    global_model.load_state_dict(new_global_weights)
    print("Server aggregated weights (FedAvg).")


    # ----- GLOBAL MODEL EVALUATION (Validation) -----
    print(f"\n----- Round {r+1} (Validation) -----")

    for cid in range(1, NUM_CLIENTS + 1):
        val_loader = client_val_loaders[cid]
        loss, report = local_eval(global_model, val_loader, DEVICE)

        current_accuracy = report['accuracy']

        print(f"\nClient {cid} Global Model Validation (Round {r+1}): Loss: {loss:.4f} | Accuracy: {current_accuracy:.4f}")

        # Print detailed, aligned report
        print_aligned_report(report, f"Class-wise Metrics (Client {cid})")


        # Check if this is the best accuracy found so far
        if current_accuracy > best_global_accuracy:
            best_global_accuracy = current_accuracy
            best_metrics_report = copy.deepcopy(report)
            best_round_info["round"] = r + 1
            best_round_info["client_id"] = cid

    print("\n" + "="*50)


print("\nFL Training Completed.")

# ----- FINAL BEST METRICS REPORT -----
print("\n" + "="*50)
print("             BEST GLOBAL MODEL METRICS             ")
print(f" (Achieved in Round {best_round_info['round']}, on Client {best_round_info['client_id']}'s validation set)")
print("="*50)

if best_metrics_report:
    print_aligned_report(best_metrics_report, "Best Model Class-wise Performance")
    print(f"\nBest Validation Accuracy: {best_global_accuracy:.4f}")
    print(f"Best Val Loss: {best_metrics_report.get('loss', 'N/A')}")
else:
    print("No validation results recorded.")

Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:00<00:00, 197MB/s]


Starting FL Training with Round-wise Validation and Best Metrics Tracking...

----- Round 1 (Training) -----
  Client 1 finished local training.
  Client 2 finished local training.
  Client 3 finished local training.
Server aggregated weights (FedAvg).

----- Round 1 (Validation) -----

Client 1 Global Model Validation (Round 1): Loss: 0.7526 | Accuracy: 0.7254

--- Class-wise Metrics (Client 1) ---
class     precision   recall    f1-score    support   
------------------------------------------------------
0            0.5172   0.5000    0.5085     30.0      
1            0.0000   0.0000    0.0000     10.0      
2            0.8750   0.9506    0.9112     162.0     
3            0.5273   0.7250    0.6105     40.0      
4            0.3333   0.3333    0.3333     24.0      
5            0.0000   0.0000    0.0000     18.0      
------------------------------------------------------
accuracy                        0.7254284       

Client 2 Global Model Validation (Round 1): Loss: 0.7116 |

In [11]:
print("\n===== Final Evaluation on Each Client =====")

for cid in range(1, NUM_CLIENTS + 1):
    _, val_loader = load_client_data(cid)
    loss, report = local_eval(global_model, val_loader, DEVICE)
    print(f"\nClient {cid} Validation:")
    print("Loss:", loss)
    print("Accuracy:", report["accuracy"])
    print(report["macro avg"])



===== Final Evaluation on Each Client =====

Client 1 Validation:
Loss: 0.769437700510025
Accuracy: 0.8450704225352113
{'precision': 0.7638091628889175, 'recall': 0.7386316872427985, 'f1-score': 0.7367114691318846, 'support': 284.0}

Client 2 Validation:
Loss: 0.5631790094905429
Accuracy: 0.8666666666666667
{'precision': 0.7623263452291139, 'recall': 0.7731818011974259, 'f1-score': 0.7666444289747458, 'support': 285.0}

Client 3 Validation:
Loss: 0.803110702170266
Accuracy: 0.8566308243727598
{'precision': 0.7412253187988481, 'recall': 0.7560268153458117, 'f1-score': 0.746134017012229, 'support': 279.0}
